In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='b9ceecdb-4a79-4e14-9eb3-3446c58d4787'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'call_3234631': 'EnMKcQFpFH0TbJ4Yb3mtuiUHCSBvUsTpSiokHuHE37PBm/niTw2m4IINIprJXjQJeC2J459BvndgjFj1u5R0sOi6R3xM1Bv8J6E6365XSRJ+r19qTq4eTniWt68RfD/f7MDXb/8xeOLaZqLWE10l5nSDlJoZ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14e11-afe9-7582-ae2f-52afbbd8f08c-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters library'}, 'id': 'call_3234631', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 24, 'total_tokens': 223,

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "code-reasoning": {
            "transport": "stdio",
            "command": "npx",
            "args": [
                "-y",
                "@mettamatt/code-reasoning",
            ],
        }
    }
)

tools = await client.get_tools()

tools = await client.get_tools()

ValueError: Configuration error: Missing 'transport' key in server configuration. Each server must include 'transport' with one of: 'stdio', 'sse', 'websocket', 'http'. Please refer to the langchain-mcp-adapters documentation for more details.

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)